In [64]:
%pip install numpy scikit-learn

Note: you may need to restart the kernel to use updated packages.


### Vectorización de texto y modelo de clasificación Naïve Bayes con el dataset 20 newsgroups

In [65]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.naive_bayes import MultinomialNB, ComplementNB
from sklearn.metrics import f1_score

# 20newsgroups por ser un dataset clásico de NLP ya viene incluido y formateado
# en sklearn
from sklearn.datasets import fetch_20newsgroups
import numpy as np

## Carga de datos

In [66]:
# cargamos los datos (ya separados de forma predeterminada en train y test)
newsgroups_train = fetch_20newsgroups(subset='train', remove=('headers', 'footers', 'quotes'))
newsgroups_test = fetch_20newsgroups(subset='test', remove=('headers', 'footers', 'quotes'))

## Vectorización

In [67]:
# instanciamos un vectorizador
# ver diferentes parámetros de instanciación en la documentación de sklearn https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html
tfidfvect = TfidfVectorizer()

In [68]:
# en el atributo `data` accedemos al texto
print(newsgroups_train.data[0])

I was wondering if anyone out there could enlighten me on this car I saw
the other day. It was a 2-door sports car, looked to be from the late 60s/
early 70s. It was called a Bricklin. The doors were really small. In addition,
the front bumper was separate from the rest of the body. This is 
all I know. If anyone can tellme a model name, engine specs, years
of production, where this car is made, history, or whatever info you
have on this funky looking car, please e-mail.


In [69]:
# con la interfaz habitual de sklearn podemos fitear el vectorizador
# (obtener el vocabulario y calcular el vector IDF)
# y transformar directamente los datos
X_train = tfidfvect.fit_transform(newsgroups_train.data)
# `X_train` la podemos denominar como la matriz documento-término

In [70]:
# recordar que las vectorizaciones por conteos son esparsas
# por ello sklearn convenientemente devuelve los vectores de documentos
# como matrices esparsas
print(type(X_train))
print(f'shape: {X_train.shape}')
print(f'Cantidad de documentos: {X_train.shape[0]}')
print(f'Tamaño del vocabulario (dimensionalidad de los vectores): {X_train.shape[1]}')

<class 'scipy.sparse._csr.csr_matrix'>
shape: (11314, 101631)
Cantidad de documentos: 11314
Tamaño del vocabulario (dimensionalidad de los vectores): 101631


In [71]:
# una vez fiteado el vectorizador, podemos acceder a atributos como el vocabulario
# aprendido. Es un diccionario que va de términos a índices.
# El índice es la posición en el vector de documento.
tfidfvect.vocabulary_['car']

25775

In [72]:
# es muy útil tener el diccionario opuesto que va de índices a términos
idx2word = {v: k for k,v in tfidfvect.vocabulary_.items()}

In [73]:
# en `y_train` guardamos los targets que son enteros
y_train = newsgroups_train.target
y_train[:10]

array([ 7,  4,  4,  1, 14, 16, 13,  3,  2,  4])

In [74]:
# hay 20 clases correspondientes a los 20 grupos de noticias
print(f'clases {np.unique(newsgroups_test.target)}')
newsgroups_test.target_names

clases [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19]


['alt.atheism',
 'comp.graphics',
 'comp.os.ms-windows.misc',
 'comp.sys.ibm.pc.hardware',
 'comp.sys.mac.hardware',
 'comp.windows.x',
 'misc.forsale',
 'rec.autos',
 'rec.motorcycles',
 'rec.sport.baseball',
 'rec.sport.hockey',
 'sci.crypt',
 'sci.electronics',
 'sci.med',
 'sci.space',
 'soc.religion.christian',
 'talk.politics.guns',
 'talk.politics.mideast',
 'talk.politics.misc',
 'talk.religion.misc']

## Similaridad de documentos

In [75]:
# Veamos similaridad de documentos. Tomemos algún documento
idx = 4811
print(newsgroups_train.data[idx])

THE WHITE HOUSE

                  Office of the Press Secretary
                   (Pittsburgh, Pennslyvania)
______________________________________________________________
For Immediate Release                         April 17, 1993     

             
                  RADIO ADDRESS TO THE NATION 
                        BY THE PRESIDENT
             
                Pittsburgh International Airport
                    Pittsburgh, Pennsylvania
             
             
10:06 A.M. EDT
             
             
             THE PRESIDENT:  Good morning.  My voice is coming to
you this morning through the facilities of the oldest radio
station in America, KDKA in Pittsburgh.  I'm visiting the city to
meet personally with citizens here to discuss my plans for jobs,
health care and the economy.  But I wanted first to do my weekly
broadcast with the American people. 
             
             I'm told this station first broadcast in 1920 when
it reported that year's presidential elec

In [76]:
# midamos la similaridad coseno con todos los documentos de train
cossim = cosine_similarity(X_train[idx], X_train)[0]

In [77]:
cossim

array([0.1382319 , 0.1067036 , 0.23029327, ..., 0.12320753, 0.08765353,
       0.04415046], shape=(11314,))

In [78]:
cosine_similarity(X_train[idx], X_train)

array([[0.1382319 , 0.1067036 , 0.23029327, ..., 0.12320753, 0.08765353,
        0.04415046]], shape=(1, 11314))

In [79]:
# podemos ver los valores de similaridad ordenados de mayor a menos
np.sort(cossim)[::-1]

array([1.        , 0.70930477, 0.67474953, ..., 0.        , 0.        ,
       0.        ], shape=(11314,))

In [80]:
# y a qué documentos corresponden
np.argsort(cossim)[::-1]

array([4811, 6635, 4253, ..., 9019, 9016, 8748], shape=(11314,))

In [81]:
# los 5 documentos más similares:
mostsim = np.argsort(cossim)[::-1][1:6]

In [82]:
# el documento original pertenece a la clase:
newsgroups_train.target_names[y_train[idx]]

'talk.politics.misc'

In [83]:
# y los 5 más similares son de las clases:
for i in mostsim:
  print(newsgroups_train.target_names[y_train[i]])

talk.politics.misc
talk.politics.misc
talk.politics.misc
talk.politics.misc
talk.politics.misc


### Modelo de clasificación Naïve Bayes

In [84]:
# es muy fácil instanciar un modelo de clasificación Naïve Bayes y entrenarlo con sklearn
clf = MultinomialNB()
clf.fit(X_train, y_train)

,alpha,1.0
,force_alpha,True
,fit_prior,True
,class_prior,None


In [85]:
# con nuestro vectorizador ya fiteado en train, vectorizamos los textos
# del conjunto de test
X_test = tfidfvect.transform(newsgroups_test.data)
y_test = newsgroups_test.target
y_pred =  clf.predict(X_test)

In [86]:
# el F1-score es una metrica adecuada para reportar desempeño de modelos de claificación
# es robusta al desbalance de clases. El promediado 'macro' es el promedio de los
# F1-score de cada clase. El promedio 'micro' es equivalente a la accuracy que no
# es una buena métrica cuando los datasets son desbalanceados
f1_score(y_test, y_pred, average='macro')

0.5854345727938506

### Consigna del desafío 1

**1**. Vectorizar documentos. Tomar 5 documentos al azar y medir similaridad con el resto de los documentos.
Estudiar los 5 documentos más similares de cada uno analizar si tiene sentido
la similaridad según el contenido del texto y la etiqueta de clasificación.

**2**. Construir un modelo de clasificación por prototipos (tipo zero-shot). Clasificar los documentos de un conjunto de test comparando cada uno con todos los de entrenamiento y asignar la clase al label del documento del conjunto de entrenamiento con mayor similaridad.

**3**. Entrenar modelos de clasificación Naïve Bayes para maximizar el desempeño de clasificación
(f1-score macro) en el conjunto de datos de test. Considerar cambiar parámteros
de instanciación del vectorizador y los modelos y probar modelos de Naïve Bayes Multinomial
y ComplementNB.

**4**. Transponer la matriz documento-término. De esa manera se obtiene una matriz
término-documento que puede ser interpretada como una colección de vectorización de palabras.
Estudiar ahora similaridad entre palabras tomando 5 palabras y estudiando sus 5 más similares. **La elección de palabras no debe ser al azar para evitar la aparición de términos poco interpretables, elegirlas "manualmente"**.


In [87]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.naive_bayes import MultinomialNB, ComplementNB
from sklearn.metrics import f1_score

# 20newsgroups por ser un dataset clásico de NLP ya viene incluido y formateado
# en sklearn
from sklearn.datasets import fetch_20newsgroups
import numpy as np

In [88]:
# cargamos los datos (ya separados de forma predeterminada en train y test)
newsgroups_train = fetch_20newsgroups(subset='train', remove=('headers', 'footers', 'quotes'))
newsgroups_test = fetch_20newsgroups(subset='test', remove=('headers', 'footers', 'quotes'))

In [89]:
# instanciamos un vectorizador
# ver diferentes parámetros de instanciación en la documentación de sklearn https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html
tfidfvect = TfidfVectorizer()

# 1 Vectorizar Documentos

In [90]:
X_train = tfidfvect.fit_transform(newsgroups_train.data)
y_train = newsgroups_train.target

In [91]:
def evaluar_similitud(ref_index, X, y, newsgroups_train, top_k=5):
    CANTIDAD_TEXTO = 100
    # Documento de referencia
    x_ref = X[ref_index].reshape(1, -1)
    ref_label = newsgroups_train.target_names[y[ref_index]]
    print("="*80)
    print(f"📌 Documento de referencia (idx={ref_index}) | Clase: {ref_label}")
    print("-"*80)
    print(newsgroups_train.data[ref_index][:CANTIDAD_TEXTO], "\n")

    # Calcular similitud coseno
    cossim = cosine_similarity(x_ref, X).ravel()
    # Obtener top_k más similares
    top_k_idx = np.argsort(-cossim)[:top_k]

    print("="*80)
    print(f"🔎 Top {top_k} documentos más similares:")
    print("="*80)

    for i, idx in enumerate(top_k_idx, start=1):
        label = newsgroups_train.target_names[y[idx]]
        print(f"[{i}] idx={idx} | Similitud={cossim[idx]:.4f} | Clase: {label}")
        print("-"*80)
        print(newsgroups_train.data[idx][:CANTIDAD_TEXTO], "\n")

    print("="*80)

In [92]:
evaluar_similitud(10,X_train,y_train, newsgroups_train, 5)

📌 Documento de referencia (idx=10) | Clase: rec.motorcycles
--------------------------------------------------------------------------------
I have a line on a Ducati 900GTS 1978 model with 17k on the clock.  Runs
very well, paint is the bro 

🔎 Top 5 documentos más similares:
[1] idx=10 | Similitud=1.0000 | Clase: rec.motorcycles
--------------------------------------------------------------------------------
I have a line on a Ducati 900GTS 1978 model with 17k on the clock.  Runs
very well, paint is the bro 

[2] idx=3543 | Similitud=0.4962 | Clase: rec.motorcycles
--------------------------------------------------------------------------------

Now you know why I am just a DOD member.  I like bikes and clubs but
the politics and other b*llsh* 

[3] idx=9171 | Similitud=0.4589 | Clase: rec.motorcycles
--------------------------------------------------------------------------------

More like those who use their backs instead of their minds to make
their living who are usually ign 

[

In [93]:
evaluar_similitud(20,X_train,y_train, newsgroups_train, 5)

📌 Documento de referencia (idx=20) | Clase: alt.atheism
--------------------------------------------------------------------------------

[...]

These don't seem like "little things" to me.  At least, they are orders
worse than the motto 

🔎 Top 5 documentos más similares:
[1] idx=20 | Similitud=1.0000 | Clase: alt.atheism
--------------------------------------------------------------------------------

[...]

These don't seem like "little things" to me.  At least, they are orders
worse than the motto 

[2] idx=10254 | Similitud=0.4432 | Clase: alt.atheism
--------------------------------------------------------------------------------


The "`little' things" above were in reference to Germany, clearly.  People
said that there were si 

[3] idx=1137 | Similitud=0.3694 | Clase: alt.atheism
--------------------------------------------------------------------------------


So, we should ban the ammunition?  Why not get rid of the guns?


It is worse than others?  The Na 

[4] idx=10779 | 

In [94]:
evaluar_similitud(653,X_train,y_train, newsgroups_train, 5)

📌 Documento de referencia (idx=653) | Clase: rec.sport.baseball
--------------------------------------------------------------------------------
Has anyone heard anything about Mel Hall this season?  I'd heard he wasn't
with the Yankees any more 

🔎 Top 5 documentos más similares:
[1] idx=653 | Similitud=1.0000 | Clase: rec.sport.baseball
--------------------------------------------------------------------------------
Has anyone heard anything about Mel Hall this season?  I'd heard he wasn't
with the Yankees any more 

[2] idx=4187 | Similitud=0.3219 | Clase: rec.sport.baseball
--------------------------------------------------------------------------------

Mel Hall signed with a Japanese team.
 

[3] idx=7394 | Similitud=0.2242 | Clase: rec.sport.baseball
--------------------------------------------------------------------------------

Mel is alive and well and playing in Japan. (The Yanks let him go because
he was asking for too muc 

[4] idx=5323 | Similitud=0.2020 | Clase: rec.spo

In [95]:
evaluar_similitud(7000,X_train,y_train, newsgroups_train, 5)

📌 Documento de referencia (idx=7000) | Clase: comp.sys.mac.hardware
--------------------------------------------------------------------------------
Dear Netters,

My sister has an Apple 12" Color Display hooked up to an LC.

Problem:  There is an a 

🔎 Top 5 documentos más similares:
[1] idx=7000 | Similitud=1.0000 | Clase: comp.sys.mac.hardware
--------------------------------------------------------------------------------
Dear Netters,

My sister has an Apple 12" Color Display hooked up to an LC.

Problem:  There is an a 

[2] idx=1533 | Similitud=0.1897 | Clase: soc.religion.christian
--------------------------------------------------------------------------------
*******
*******  This is somewhat long, but pleas read it!!!!!!!!!!!!!!!!!
*******



Boy am i glad  

[3] idx=386 | Similitud=0.1744 | Clase: sci.electronics
--------------------------------------------------------------------------------
Sci.E(E) netters:

I am setting out to build and market a small electronic device 

In [96]:
evaluar_similitud(8556,X_train,y_train, newsgroups_train, 5)

📌 Documento de referencia (idx=8556) | Clase: sci.crypt
--------------------------------------------------------------------------------


        But that's just the problem. There is no such thing as
        "MIME-Formatted". By analog 

🔎 Top 5 documentos más similares:
[1] idx=8556 | Similitud=1.0000 | Clase: sci.crypt
--------------------------------------------------------------------------------


        But that's just the problem. There is no such thing as
        "MIME-Formatted". By analog 

[2] idx=5976 | Similitud=0.3409 | Clase: sci.crypt
--------------------------------------------------------------------------------
1. Do a straight encryption of your keyrings and put the
        results with misleading names somew 

[3] idx=8478 | Similitud=0.1963 | Clase: talk.politics.mideast
--------------------------------------------------------------------------------


Peter,

I believe this is your most succinct post to date. Since you have nothing
to say, you say  

[4] idx=7

Al analizar los diferentes resultados, podemos notar como esta métrica de similitud nos puede ayudar a obtener una primera iteración en la detección de documentos similares. Para cada indice elegido (primer atributo de la función desarrollada) obtenemos los 5 documentos mas similares. Como era de esperarse, con esta configuración siempre el documento con mayor similitud será el mismo que se está evaluando, por lo cual se puede obtener un mejor resultado quitando este de la obtención de similitud.

A su vez, vemos como si bien suele estar acertada la descripción, muchas veces los documentos mas similares no pertenecen a la misma clase. Por ejemplo, la clase del primer documento (distinto del mismo docuemnto) en el indice 7000 no posee la misma clase: mientras que el documento es de la clase **comp.sys.mac.hardware**, el que presenta mayor similitud es de la clase **soc.religion.christian**

# Clasificador Zero Shot

In [108]:
tfidfvect = TfidfVectorizer()

X_train = tfidfvect.fit_transform(newsgroups_train.data)
y_train = newsgroups_train.target

X_test = tfidfvect.transform(newsgroups_test.data)
y_test = newsgroups_test.target

In [111]:
def zero_shot_predictor(X_train, X_test, y_train, y_test):

    similar_idx = cosine_similarity(X_test, X_train).argmax(axis=1)
    y_pred = y_train[similar_idx]
    
    f1_res = f1_score(y_test, y_pred, average='macro')

    print("f1 score:",f1_res)

    return y_pred

In [ ]:
y_pred = zero_shot_predictor(X_train, X_test, y_train, y_test)

f1 score: 0.5049911553681621


array([ 0, 19, 17, ..., 17, 12, 15], shape=(7532,))

Podemos observar como se realizó el clasificador zero shot de forma vectorizada: al utilizar arrays de numpy, podemos realizar todas las clasificaciones de forma simultanea. Al analizar los resultados, podemos determinar dos conclusiones:

1. El clasificador tiene mejor desempeño que una clasificador que adivina, llegando a un buen resultado para una primera iteración. Para 20 clases, si el clasificador estuviese adivinando tendríamos un f1 score de 0.05, mientras que nuestro valor es de 0.5. 
1. Esta metodología puede resultar algo ineficiente, ya que para realizar predicciones, deberiamos mantener el dataset completo de entrenamiento. Esto puede no resultar ideal para datasets de tamaño significativo.

# Clasificador Naive Bayes

**3**. Entrenar modelos de clasificación Naïve Bayes para maximizar el desempeño de clasificación
(f1-score macro) en el conjunto de datos de test. Considerar cambiar parámteros
de instanciación del vectorizador y los modelos y probar modelos de Naïve Bayes Multinomial
y ComplementNB.

Para este ejercicio, buscamos hacer una configuración que sea muy customizable, que nos permita encontrar el modelo e hiperparámetros con mejor desempeño. Utilizaremos para esta tarea OPTUNA

In [ ]:
tfidfvect = TfidfVectorizer()

X_train = tfidfvect.fit_transform(newsgroups_train.data)
y_train = newsgroups_train.target

X_test = tfidfvect.transform(newsgroups_test.data)
y_test = newsgroups_test.target

Definimos la función con rangos de parámetros de optuna en los cuales se entrenará:
- Alpha
- Fit Prior
- Modelo: **MultinomialNB** o **ComplementNB**

In [3]:
import optuna
import numpy as np
from sklearn.naive_bayes import MultinomialNB, ComplementNB
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import f1_score
import tqdm
# ----- Config -----
N_TRIALS = 40
SEED = 42
CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
SCORING = "f1_macro"

def objective(trial, model_type: str):
    # Espacio de búsqueda común
    alpha = trial.suggest_float("alpha", 1e-3, 10.0, log=True)
    fit_prior = trial.suggest_categorical("fit_prior", [True, False])

    if model_type == "mnb":
        model = MultinomialNB(alpha=alpha, fit_prior=fit_prior)
    elif model_type == "cnb":
        norm = trial.suggest_categorical("norm", [True, False])
        model = ComplementNB(alpha=alpha, fit_prior=fit_prior, norm=norm)
    else:
        raise ValueError("model_type must be 'mnb' or 'cnb'")

    # CV en train con F1 macro
    scores = cross_val_score(model, X_train, y_train, cv=CV, scoring=SCORING, n_jobs=-1)
    return scores.mean()


## MultinomialNB

Optimización de hiperparámetros

In [ ]:
# ----- Optimización MNB -----
study_mnb = optuna.create_study(direction="maximize", study_name="MultinomialNB")
study_mnb.optimize(lambda t: objective(t, "mnb"), n_trials=N_TRIALS, show_progress_bar=False)
print("\n[MNB] Best CV F1-macro:", study_mnb.best_value)
print("[MNB] Best params:", study_mnb.best_params)

[I 2025-09-03 20:27:19,568] A new study created in memory with name: MultinomialNB
[I 2025-09-03 20:27:20,165] Trial 0 finished with value: 0.5383428691794241 and parameters: {'alpha': 3.0556997986931624, 'fit_prior': True}. Best is trial 0 with value: 0.5383428691794241.
[I 2025-09-03 20:27:20,778] Trial 1 finished with value: 0.6395713704814503 and parameters: {'alpha': 0.9966863892159104, 'fit_prior': False}. Best is trial 1 with value: 0.6395713704814503.
[I 2025-09-03 20:27:21,381] Trial 2 finished with value: 0.7540926336546477 and parameters: {'alpha': 0.0022215475727044294, 'fit_prior': True}. Best is trial 2 with value: 0.7540926336546477.
[I 2025-09-03 20:27:22,025] Trial 3 finished with value: 0.7547256656941451 and parameters: {'alpha': 0.005634291540364055, 'fit_prior': True}. Best is trial 3 with value: 0.7547256656941451.
[I 2025-09-03 20:27:22,442] Trial 4 finished with value: 0.6439827879594848 and parameters: {'alpha': 0.9295885429757599, 'fit_prior': False}. Best is 


[MNB] Best CV F1-macro: 0.7599762963768077
[MNB] Best params: {'alpha': 0.00770944569868003, 'fit_prior': False}
[MNB] Test F1-macro: 0.6875701959376942


Entrenamiento con mejores hiperparámetros y evaluación de resultados

In [ ]:
best_mnb = MultinomialNB(**study_mnb.best_params).fit(X_train, y_train)
y_pred_mnb = best_mnb.predict(X_test)
print("[MNB] Test F1-macro:", f1_score(y_test, y_pred_mnb, average="macro"))

## ComplementNB

Optimización de hiperparámetros

In [ ]:
# ----- Optimización CNB -----
study_cnb = optuna.create_study(direction="maximize", study_name="ComplementNB")
study_cnb.optimize(lambda t: objective(t, "cnb"), n_trials=N_TRIALS, show_progress_bar=False)
print("\n[CNB] Best CV F1-macro:", study_cnb.best_value)
print("[CNB] Best params:", study_cnb.best_params)

[I 2025-09-03 20:27:47,314] A new study created in memory with name: ComplementNB
[I 2025-09-03 20:27:47,981] Trial 0 finished with value: 0.71412887721942 and parameters: {'alpha': 4.396039076487217, 'fit_prior': False, 'norm': False}. Best is trial 0 with value: 0.71412887721942.
[I 2025-09-03 20:27:48,583] Trial 1 finished with value: 0.7225438360432712 and parameters: {'alpha': 3.37137783710806, 'fit_prior': False, 'norm': False}. Best is trial 1 with value: 0.7225438360432712.
[I 2025-09-03 20:27:49,226] Trial 2 finished with value: 0.7601878971771396 and parameters: {'alpha': 0.07867774039646314, 'fit_prior': False, 'norm': False}. Best is trial 2 with value: 0.7601878971771396.
[I 2025-09-03 20:27:49,678] Trial 3 finished with value: 0.7536655891863782 and parameters: {'alpha': 0.3266671094214983, 'fit_prior': False, 'norm': True}. Best is trial 2 with value: 0.7601878971771396.
[I 2025-09-03 20:27:50,125] Trial 4 finished with value: 0.7165733756270221 and parameters: {'alpha':


[CNB] Best CV F1-macro: 0.7638795012356314
[CNB] Best params: {'alpha': 0.1818067530598363, 'fit_prior': True, 'norm': False}
[CNB] Test F1-macro: 0.6997838327093598


Entrenamiento con mejores hiperparámetros y evaluación de resultados

In [ ]:
# Entrenar y evaluar en test
best_cnb = ComplementNB(**study_cnb.best_params).fit(X_train, y_train)
y_pred_cnb = best_cnb.predict(X_test)
print("[CNB] Test F1-macro:", f1_score(y_test, y_pred_cnb, average="macro"))